In [5]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.io import decode_image
from torchvision.transforms import v2
import pandas as pd
from PIL import Image

In [6]:
class StatementDataset(Dataset):
    def __init__(self, annotations_file: str, image_dir: str, image_transform=None, text_transform=None,
                 target_transform=None):
        self.labels = pd.read_csv(annotations_file)
        self.image_dir = image_dir
        self.image_transform = image_transform
        self.text_transform = text_transform
        self.target_transform = target_transform

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        img_path = os.path.join(self.image_dir, self.labels.iloc[index, 0])
        img = decode_image(img_path)
        label = torch.tensor(self.labels.iloc[index, 1])

        if self.image_transform:
            img = self.image_transform(img)
        if self.text_transform:
            pass
        if self.target_transform:
            label = self.target_transform(label)

        return img, label


class OneHotTransform:
    """Custom transform to convert labels to one hot vectors"""

    def __init__(self, num_classes: int):
        self.num_classes = num_classes

    def __call__(self, label):
        tensor = torch.zeros(self.num_classes)
        tensor[label] = 1
        return tensor


class SquarePadTransform:
    """Custom transform to pad images to square with specified size"""

    def __init__(self, target_size=1568, fill=0):
        self.target_size = target_size
        self.fill = fill

    def __call__(self, img):
        # img should be a PIL Image or Tensor
        if isinstance(img, Image.Image):
            w, h = img.size
        else:  # Tensor
            c, h, w = img.shape

        # Calculate padding needed
        pad_h = max(0, self.target_size - h)
        pad_w = max(0, self.target_size - w)

        # Pad equally on both sides (or as close as possible)
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top

        # Use v2.Pad
        padding_transform = v2.Pad(
            padding=[pad_left, pad_top, pad_right, pad_bottom],
            fill=self.fill
        )

        return padding_transform(img)


In [7]:
image_transforms = v2.Compose([
    v2.ToImage(),  # Convert to tensor image
    v2.ToDtype(torch.float32, scale=True),  # Convert to float and scale to [0, 1]
    SquarePadTransform(target_size=1568, fill=0),  # Pad to 1568x1568
    v2.Grayscale()
])
target_transforms = v2.Compose([OneHotTransform(num_classes=3)])

data = StatementDataset("dataset\\labels.csv", image_dir="dataset\\images", image_transform=image_transforms)

train_len = int(0.8 * len(data))
test_len = len(data) - train_len
train_dataset, test_dataset = random_split(data, [train_len, test_len])

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False, num_workers=2)


In [8]:
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler


# Residual Block
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Skip connection
        self.skip = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.skip(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out


# Main Model
class FinancialStatementCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(FinancialStatementCNN, self).__init__()

        # Stage 1: Aggressive Initial Downsampling
        # Input: 1x1568x1568 -> Output: 64x523x523
        self.stage1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=7, stride=3, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)  # -> 64x261x261
        )

        # Stage 2: Early Feature Extraction (3 blocks, 128 filters)
        # Output: 128x261x261
        self.stage2 = nn.Sequential(
            ResidualBlock(64, 128, stride=1),
            ResidualBlock(128, 128, stride=1),
            ResidualBlock(128, 128, stride=1)
        )

        # Stage 3: Mid-level Patterns (4 blocks, 256 filters)
        # Output: 256x131x131
        self.stage3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256, stride=1),
            ResidualBlock(256, 256, stride=1),
            ResidualBlock(256, 256, stride=1)
        )

        # Stage 4: High-level Document Structure (6 blocks, 512 filters)
        # Output: 512x66x66
        self.stage4 = nn.Sequential(
            ResidualBlock(256, 512, stride=2),
            ResidualBlock(512, 512, stride=1),
            ResidualBlock(512, 512, stride=1),
            ResidualBlock(512, 512, stride=1),
            ResidualBlock(512, 512, stride=1),
            ResidualBlock(512, 512, stride=1)
        )

        # Stage 5: Abstract Layout Features (3 blocks, 512 filters)
        # Output: 512x33x33
        self.stage5 = nn.Sequential(
            ResidualBlock(512, 512, stride=2),
            ResidualBlock(512, 512, stride=1),
            ResidualBlock(512, 512, stride=1)
        )

        # Stage 6: Classification Head
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.stage5(x)
        x = self.global_avg_pool(x)
        x = self.classifier(x)
        return x


# Training Loop
def train_model(model, train_loader, test_loader, num_epochs=50, device='cuda'):
    """
    Training loop for the financial statement classifier

    Args:
        model: The CNN model
        train_loader: DataLoader for training data
        test_loader: DataLoader for test/validation data
        num_epochs: Number of training epochs
        device: 'cuda' or 'cpu'
    """

    model = model.to(device)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    # Mixed precision training
    scaler = GradScaler()

    best_acc = 0.0

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            # Convert one-hot to class indices
            labels = labels.to(device)
            optimizer.zero_grad()

            # Mixed precision forward pass
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            # Backward pass with gradient scaling
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Statistics
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            print(f'Epoch [{epoch + 1}/{num_epochs}], Step [{batch_idx + 1}/{len(train_loader)}], '
                  f'Loss: {loss.item():.4f}')

        train_acc = 100 * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)

        # Validation phase
        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                test_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        test_acc = 100 * test_correct / test_total
        avg_test_loss = test_loss / len(test_loader)

        # Update learning rate
        scheduler.step(avg_test_loss)

        print(f'\nEpoch [{epoch + 1}/{num_epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'Test Loss: {avg_test_loss:.4f}, Test Acc: {test_acc:.2f}%')
        print('-' * 60)

        # Save best model
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'test_acc': test_acc,
                'test_loss': avg_test_loss
            }, 'best_financial_cnn_model.pth')
            print(f'Best model saved with accuracy: {best_acc:.2f}%\n')

    print(f'Training completed! Best test accuracy: {best_acc:.2f}%')
    return model


# Usage Example
if __name__ == "__main__":
    # Initialize model
    model = FinancialStatementCNN(num_classes=3)

    # Print model summary
    # print("Model Architecture:")
    # print(model)
    # print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
    # print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # Check device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nUsing device: {device}")
    # Train the model (assuming train_loader and test_loader are defined)

    checkpoint = torch.load('best_financial_cnn_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])



Model Architecture:
FinancialStatementCNN(
  (stage1): Sequential(
    (0): Conv2d(1, 64, kernel_size=(7, 7), stride=(3, 3), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (stage2): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (skip): Sequential(
        (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_

In [22]:
model = model.to(device)
model.eval()
with torch.no_grad():
    item = data[72935]
    images = item[0].unsqueeze(1).to(device)
    outputs = model(images)
    soft = torch.softmax(outputs.data, dim=1)
    _, predicted = torch.max(soft,1)
    print(predicted, item[1])

tensor([2], device='cuda:0') tensor(2)
